# Stage 1 Momentum Single-Factor Showcase

This notebook is a presentation-oriented walkthrough of the current Stage 1 A-share single-factor pipeline.
It does **not** replace modular code under `src/`; it only demonstrates outputs and analysis steps.

## 1) Brief project overview

Current focus: momentum single-factor research with strict anti-look-ahead handling.

- Data loading/cleaning and schema normalization
- Universe construction (alive + listing-age filters)
- Momentum factor and preprocessing
- Signal lagging and forward return label
- IC and quantile backtest showcase

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure project root is importable when notebook is run from `notebooks/`
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.pipeline import run_pipeline
from src.config import PipelineConfig

plt.style.use('seaborn-v0_8-whitegrid')
print('Project root for imports:', PROJECT_ROOT)

## 2) Data loading / sample preview

In [ ]:
output_path = Path('output/pipeline_output.parquet')

if output_path.exists():
    panel = pd.read_parquet(output_path)
    source_msg = f'Loaded existing output: {output_path}'
else:
    panel = run_pipeline(PipelineConfig())
    output_path.parent.mkdir(parents=True, exist_ok=True)
    panel.to_parquet(output_path, index=False)
    source_msg = f'Ran pipeline and saved: {output_path}'

print(source_msg)
print(panel.shape)
panel.head()

## 3) Momentum factor examples

In [ ]:
cols = [
    'date', 'stock_code', 'industry', 'close',
    'mom_raw', 'mom_ind_neutral', 'signal', 'fwd_1d_return', 'tradable'
]
existing_cols = [c for c in cols if c in panel.columns]
panel[existing_cols].head(10)

## 4) Preprocessing examples

In [ ]:
daily_stats = panel.groupby('date')[['mom_raw', 'mom_ind_neutral']].agg(['mean', 'std']).tail(5)
daily_stats

In [ ]:
tmp = panel.loc[panel['tradable'] & panel['signal'].notna(), ['date', 'signal']]
tmp['signal_rank_pct'] = tmp.groupby('date')['signal'].rank(pct=True)
tmp.head()

## 5) IC analysis results

In [ ]:
eval_df = panel.loc[panel['tradable'] & panel['signal'].notna() & panel['fwd_1d_return'].notna(),
                    ['date', 'signal', 'fwd_1d_return']].copy()

ic_by_date = eval_df.groupby('date').apply(
    lambda x: x['signal'].corr(x['fwd_1d_return'], method='spearman')
).rename('ic').dropna()

ic_mean = ic_by_date.mean()
ic_std = ic_by_date.std(ddof=0)
icir = np.nan if ic_std == 0 else ic_mean / ic_std

pd.DataFrame({'metric': ['IC Mean', 'IC Std', 'ICIR'], 'value': [ic_mean, ic_std, icir]})

## 6) Quantile backtest results

In [ ]:
nq = 5
qdf = eval_df.copy()

def _safe_qcut(s, q=nq):
    if s.nunique() < q:
        return pd.Series(np.nan, index=s.index)
    return pd.qcut(s, q=q, labels=False, duplicates='drop') + 1

qdf['quantile'] = qdf.groupby('date')['signal'].transform(_safe_qcut)
qret = qdf.dropna(subset=['quantile']).groupby(['date', 'quantile'])['fwd_1d_return'].mean().unstack()
qret.columns = [f'Q{int(c)}' for c in qret.columns]
qret['Q5-Q1'] = qret.get('Q5', np.nan) - qret.get('Q1', np.nan)
qret.tail()

## 7) Key plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ic_by_date.cumsum().plot(ax=axes[0], color='tab:blue', lw=1.5)
axes[0].set_title('Cumulative IC')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Cumulative IC')

(1 + qret[['Q1','Q5','Q5-Q1']].fillna(0)).cumprod().plot(ax=axes[1], lw=1.5)
axes[1].set_title('Quantile / Long-Short Cumulative Return')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Cumulative Return (gross)')

plt.tight_layout()
plt.show()

## 8) Short concluding remarks

- Stage 1 currently provides a clean, modular momentum single-factor pipeline with anti-look-ahead signal handling.
- Early diagnostics are available through IC and quantile analysis in this notebook.
- Next stages will focus on multi-factor expansion, prediction modeling, and portfolio optimization.